# Week 31, Day 3: Rich comparisons

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w031d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Runner:
    def __init__(self, name, seconds):
        self.name = name
        self.seconds = seconds

    def __lt__(self, other):
        return self.seconds < other.seconds

    def __repr__(self):
        return self.name

runners = [Runner("Ada", 62), Runner("Bo", 58), Runner("Chi", 60)]
print(sorted(runners), min(runners), max(runners))
print(Runner("x", 1) > Runner("y", 2))

In [ ]:
class Runner:
    def __init__(self, name, seconds):
        self.name, self.seconds = name, seconds

    def __eq__(self, other):
        if not isinstance(other, Runner):
            return NotImplemented
        return self.seconds == other.seconds

    def __lt__(self, other):
        if not isinstance(other, Runner):
            return NotImplemented
        return self.seconds < other.seconds

a, b = Runner("Ada", 60), Runner("Bo", 60)
print(a == b, a < b, a > b)
try:
    print(a <= b)
except TypeError as err:
    print("TypeError:", err)
try:
    print(a < 61)
except TypeError as err:
    print("TypeError:", err)

In [ ]:
class Version:
    def __init__(self, text):
        self.text = text
        self.parts = tuple(int(p) for p in text.split("."))

    def _key(self):
        return self.parts

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._key() == other._key()

    def __hash__(self):
        return hash(self._key())

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._key() < other._key()

    def __repr__(self):
        return f"Version({self.text!r})"

vs = [Version("1.10"), Version("1.9.2"), Version("1.9"), Version("0.12")]
print(sorted(vs), max(vs))
print(Version("1.10") > Version("1.9"), "1.10" > "1.9")

In [ ]:
from functools import total_ordering

@total_ordering
class Grade:
    ORDER = ["F", "E", "D", "C", "B", "A", "A*"]

    def __init__(self, letter):
        if letter not in Grade.ORDER:
            raise ValueError(f"no grade {letter!r}")
        self.letter = letter

    def _key(self):
        return Grade.ORDER.index(self.letter)

    def __eq__(self, other):
        if not isinstance(other, Grade):
            return NotImplemented
        return self._key() == other._key()

    def __hash__(self):
        return hash(self._key())

    def __lt__(self, other):
        if not isinstance(other, Grade):
            return NotImplemented
        return self._key() < other._key()

    def __repr__(self):
        return f"Grade({self.letter!r})"

results = [Grade("B"), Grade("A*"), Grade("C"), Grade("A")]
print(sorted(results, reverse=True))
print(Grade("A") >= Grade("B"), Grade("C") <= Grade("C"), Grade("D") > Grade("E"))
print(max(results), Grade("B") in {Grade("B")})

In [ ]:
from operator import attrgetter

class Book:
    def __init__(self, title, author, year):
        self.title, self.author, self.year = title, author, year

    def __repr__(self):
        return self.title

books = [Book("Small Island", "Levy", 2004), Book("Aké", "Soyinka", 1981),
         Book("Kindred", "Butler", 1979), Book("Purple Hibiscus", "Adichie", 2003)]
print(sorted(books, key=attrgetter("year")))
print(sorted(books, key=lambda b: b.author))
print(min(books, key=lambda b: len(b.title)), max(books, key=attrgetter("year")))

**Predict before you run:** what will this print?



In [ ]:
class Tag:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name

    def __lt__(self, other):
        return self.name.lower() < other.name.lower()

a, b = Tag("python"), Tag("Python")
print(a == b, a < b, b < a)

> **Key idea.** `<`, `<=`, `>` and `>=` call `__lt__`, `__le__`, `__gt__` and `__ge__`; return `NotImplemented` for other types. Python reflects comparisons (`a > b` tries `b < a`) but never builds `<=` from `<` and `==`; `@functools.total_ordering` does, given `__eq__` and one ordering method. Compare a key tuple in `__eq__`, `__hash__` and `__lt__` so they always agree, negating numbers to reverse part of an order. Give a class an order only when there's one obvious one; otherwise sort with `key=` and `operator.attrgetter`.

### Exercise: Comparing money

Give `Money` an `__eq__`, a `__hash__`, an `__lt__` and an `__le__`, all comparing pence and all returning `NotImplemented` for anything that isn't `Money`. Don't use `total_ordering` here: check that `>` and `>=` still work, by reflection.

In [ ]:
class Money:
    def __init__(self, pence):
        self.pence = pence

    def __repr__(self):
        return f"Money({self.pence})"

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_compare():\n    \"comparisons\"\n    M = ns[\"Money\"]\n    assert M(5) < M(7) and M(7) > M(5) and M(5) <= M(5) and M(5) >= M(5) and not M(7) < M(7)\n    assert M(3) == M(3) and M(3) != M(4) and len({M(3), M(3)}) == 1\n    assert sorted([M(300), M(5), M(120)]) == [M(5), M(120), M(300)] and max([M(1), M(9), M(4)]) == M(9)\n    assert \"__gt__\" not in vars(M) and \"__ge__\" not in vars(M), \"let reflection handle > and >=\"\n    for f in [lambda: M(5) < 6, lambda: M(5) <= 6, lambda: 6 > M(5)]:\n        try:\n            f()\n            assert False, \"comparing Money with a number should raise TypeError\"\n        except TypeError:\n            pass")

### Exercise: Ranking cards

Make `Card` fully ordered with `total_ordering`: by rank first (`2` lowest, `A` highest), and for equal ranks by suit in the order `♣ ♦ ♥ ♠` (lowest to highest). Cards are equal only when rank and suit both match; equal cards hash the same. Then write `best(hand)`, the highest card in a hand, and `in_order(hand)`, the hand sorted lowest first.

In [ ]:
from functools import total_ordering

class Card:
    RANKS = ["2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K", "A"]
    SUITS = ["♣", "♦", "♥", "♠"]

    def __init__(self, rank, suit):
        if rank not in Card.RANKS or suit not in Card.SUITS:
            raise ValueError(f"no such card: {rank}{suit}")
        self.rank, self.suit = rank, suit

    def __repr__(self):
        return self.rank + self.suit

    ___

def best(hand):
    ___

def in_order(hand):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"ordering\"\n    C = ns[\"Card\"]\n    assert C(\"10\", \"\u2663\") > C(\"9\", \"\u2660\") and C(\"Q\", \"\u2660\") > C(\"Q\", \"\u2665\") and C(\"A\", \"\u2663\") > C(\"K\", \"\u2660\")\n    assert C(\"2\", \"\u2666\") <= C(\"2\", \"\u2666\") and C(\"3\", \"\u2663\") >= C(\"2\", \"\u2660\") and C(\"J\", \"\u2665\") == C(\"J\", \"\u2665\") and C(\"J\", \"\u2665\") != C(\"J\", \"\u2666\")\n    hand = [C(\"Q\", \"\u2665\"), C(\"10\", \"\u2660\"), C(\"Q\", \"\u2663\"), C(\"2\", \"\u2666\"), C(\"A\", \"\u2666\")]\n    assert repr(in_order(hand)) == \"[2\u2666, 10\u2660, Q\u2663, Q\u2665, A\u2666]\", f\"got {in_order(hand)}\"\n    assert best(hand) == C(\"A\", \"\u2666\") and len({C(\"K\", \"\u2660\"), C(\"K\", \"\u2660\")}) == 1\n    assert C(\"K\", \"\u2660\").__lt__(\"K\") is NotImplemented")

### Exercise: School terms

A school year like `"2025/26"` has three terms in order: `autumn`, `spring`, `summer`. Write `Term(year, name)`, where `year` looks like `"2025/26"` (the second part is the next year's last two digits, so `"2025/27"` is wrong) and `name` is one of the three terms; otherwise `ValueError`. Terms are ordered by year, then by term, are equal and hash the same when both match, and use `total_ordering`. `repr` gives `Term('2025/26', 'spring')`. `next()` returns the following term (after summer comes autumn of the next year: `Term("2025/26", "summer").next()` is `Term('2026/27', 'autumn')`).

In [ ]:
from functools import total_ordering

class Term:
    NAMES = ["autumn", "spring", "summer"]
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_terms():\n    \"terms\"\n    T = ns[\"Term\"]\n    a, b, c = T(\"2025/26\", \"summer\"), T(\"2026/27\", \"autumn\"), T(\"2025/26\", \"autumn\")\n    assert c < a < b and b > a and a >= a and sorted([b, a, c]) == [c, a, b]\n    assert T(\"2025/26\", \"spring\") == T(\"2025/26\", \"spring\") and len({a, T(\"2025/26\", \"summer\")}) == 1\n    assert repr(a.next()) == \"Term('2026/27', 'autumn')\" and a.next() == b and c.next() == T(\"2025/26\", \"spring\")\n    assert T(\"1999/00\", \"summer\").next() == T(\"2000/01\", \"autumn\")\n    for bad in [(\"2025/27\", \"autumn\"), (\"2025\", \"autumn\"), (\"2025/26\", \"winter\"), (\"25/26\", \"autumn\"), (\"abcd/ef\", \"spring\")]:\n        try:\n            T(*bad)\n            assert False, f\"Term{bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Sorting with keys

`Player` has no ordering on purpose: players can be ranked in different ways. Using key functions (no comparison methods), write:

- `by_goals(players)`: most goals first; ties by fewer games played; then by name A to Z;
- `youngest(players)`: the youngest player (the first one listed if several share the lowest age), or `None` for an empty list.

In [ ]:
class Player:
    def __init__(self, name, age, goals, games):
        self.name, self.age, self.goals, self.games = name, age, goals, games

    def __repr__(self):
        return self.name

def by_goals(players):
    ___

def youngest(players):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"rankings\"\n    P = ns[\"Player\"]\n    ps = [P(\"Osimhen\", 26, 20, 30), P(\"Kane\", 31, 20, 28), P(\"Saka\", 23, 12, 35), P(\"Ade\", 23, 12, 35), P(\"Lookman\", 27, 3, 10)]\n    assert repr(by_goals(ps)) == \"[Kane, Osimhen, Ade, Saka, Lookman]\", f\"got {by_goals(ps)}\"\n    assert youngest(ps).name == \"Saka\" and youngest([]) is None and by_goals([]) == []\n    assert \"__lt__\" not in vars(P)")

### Bug hunt: No less-or-equal

Checking whether a parcel is within the weight limit crashes with `TypeError: '<=' not supported between instances of 'Weight' and 'Weight'`. Fix the class so it prints `True False`, without writing `__le__` by hand.

In [ ]:
from functools import total_ordering

class Weight:
    def __init__(self, grams):
        self.grams = grams

    def __eq__(self, other):
        if not isinstance(other, Weight):
            return NotImplemented
        return self.grams == other.grams

    def __lt__(self, other):
        if not isinstance(other, Weight):
            return NotImplemented
        return self.grams < other.grams

limit = Weight(2000)
print(Weight(2000) <= limit, Weight(2500) <= limit)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"within the limit\"\n    expect_output(\"True False\")\n    assert \"__le__\" in vars(ns[\"Weight\"]) and \"def __le__\" not in source, \"use total_ordering\"")

### Bug hunt: Quietly False

Comparing a `Price` with a plain number should be refused, so this should print `can't compare`. Instead it prints `False`. Fix `__lt__`.

In [ ]:
class Price:
    def __init__(self, pence):
        self.pence = pence

    def __lt__(self, other):
        if not isinstance(other, Price):
            return False
        return self.pence < other.pence

try:
    print(Price(500) < 600)
except TypeError:
    print("can't compare")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"refused\"\n    expect_output(\"can't compare\")\n    P = ns[\"Price\"]\n    assert P(1) < P(2) and not P(2) < P(1)")

### Bug hunt: Slowest first

The race results should list the fastest time first, `[Bo 58, Chi 60, Ada 62]`, but they come out the other way round. Fix the comparison, not the call to `sorted`.

In [ ]:
class Time:
    def __init__(self, name, seconds):
        self.name, self.seconds = name, seconds

    def __lt__(self, other):
        if not isinstance(other, Time):
            return NotImplemented
        return self.seconds > other.seconds

    def __repr__(self):
        return f"{self.name} {self.seconds}"

print(sorted([Time("Ada", 62), Time("Bo", 58), Time("Chi", 60)]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fastest first\"\n    expect_output(\"[Bo 58, Chi 60, Ada 62]\")")

### Puzzle: An ordered floor number

Arrange the lines so that `Floor` is fully ordered by its number. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from functools import total_ordering

class A:
    def __init__(self, v):
        self.v = v
    def __lt__(self, other):
        return self.v < other.v

@total_ordering
class B:
    def __init__(self, v):
        self.v = v
    def __eq__(self, other):
        return self.v == other.v
    def __lt__(self, other):
        return self.v < other.v

p = ___   # A(1) < A(2)
q = ___   # A(1) > A(2)
r = ___   # B(2) >= B(2)
s = ___   # max([B(3), B(9), B(4)]).v
t = ___   # (2, "b") < (2, "a")
u = ___   # "10" < "9"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (True, False, True), \"p to r\"\n    assert (s, t, u) == (9, False, True)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: finding free time

Write `Clock(text)` for a time of day, and use it to find gaps in a busy day.

- `Clock("9:05")` and `Clock("09:05")` are the same time: hours 0 to 23, a colon, then exactly two digits of minutes 0 to 59; anything else raises `ValueError`. `str` gives `09:05` and `repr` gives `Clock('09:05')`.
- Clocks are equal, hash the same and are fully ordered by time of day (use `total_ordering`), and return `NotImplemented` when compared with anything that isn't a `Clock`.
- `later - earlier` (two clocks) gives the difference in minutes, as an `int` (negative if the left one is earlier). `clock + minutes` (an `int`) gives a new `Clock`, raising `ValueError` if it would go past 23:59 or before 00:00.
- `free_slots(busy, start, end)`: `busy` is a list of `(begin, finish)` pairs of clocks, in any order, which may overlap or reach outside the day's window. Return a sorted list of `(begin, finish)` pairs for every free gap between `start` and `end`. A gap of 0 minutes isn't a gap.

In [ ]:
import re
from functools import total_ordering

class Clock:
    ___

def free_slots(busy, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def C(t):\n    return ns[\"Clock\"](t)\ndef test_clock():\n    \"clocks\"\n    a = C(\"9:05\")\n    assert a == C(\"09:05\") and str(a) == \"09:05\" and repr(a) == \"Clock('09:05')\" and len({a, C(\"09:05\")}) == 1\n    assert C(\"9:00\") < C(\"10:00\") <= C(\"10:00\") and C(\"23:59\") > C(\"0:00\") and C(\"12:30\") >= C(\"12:29\")\n    assert C(\"10:15\") - C(\"9:05\") == 70 and C(\"9:05\") - C(\"10:15\") == -70 and C(\"9:05\") + 70 == C(\"10:15\") and C(\"1:00\") + -60 == C(\"0:00\")\n    assert sorted([C(\"13:00\"), C(\"8:30\"), C(\"9:00\")]) == [C(\"8:30\"), C(\"9:00\"), C(\"13:00\")]\n    for bad in [\"24:00\", \"9:60\", \"9:5\", \"0905\", \"\", \"-1:00\"]:\n        try:\n            C(bad)\n            assert False, f\"Clock({bad!r}) should raise ValueError\"\n        except ValueError:\n            pass\n    for f in [lambda: C(\"23:30\") + 30, lambda: C(\"0:10\") + -11]:\n        try:\n            f()\n            assert False, \"going past the day should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        C(\"9:00\") < \"10:00\"\n        assert False, \"comparing with a string should raise TypeError\"\n    except TypeError:\n        pass\ndef test_free():\n    \"free slots\"\n    busy = [(C(\"9:30\"), C(\"10:00\")), (C(\"9:00\"), C(\"9:45\")), (C(\"11:00\"), C(\"11:30\")), (C(\"13:00\"), C(\"14:00\"))]\n    got = free_slots(busy, C(\"8:30\"), C(\"12:00\"))\n    assert [(str(a), str(b)) for a, b in got] == [(\"08:30\", \"09:00\"), (\"10:00\", \"11:00\"), (\"11:30\", \"12:00\")], f\"got {got}\"\n    assert [(str(a), str(b)) for a, b in free_slots([], C(\"9:00\"), C(\"17:00\"))] == [(\"09:00\", \"17:00\")]\n    assert free_slots([(C(\"8:00\"), C(\"18:00\"))], C(\"9:00\"), C(\"17:00\")) == []\n    tight = [(C(\"9:00\"), C(\"10:00\")), (C(\"10:00\"), C(\"11:00\"))]\n    assert [(str(a), str(b)) for a, b in free_slots(tight, C(\"9:00\"), C(\"12:00\"))] == [(\"11:00\", \"12:00\")]")

## Warm-up

`sorted`, `min` and `max` only ever use `<`, so one method was enough for all three. And `a > b` worked without a `__gt__`: Python turned it round and asked `b < a` instead. Today covers the rules behind both.

## The six comparisons

Each comparison operator calls its own special method:

| You write | Python calls | If that returns `NotImplemented`, Python tries |
|---|---|---|
| `a < b` | `a.__lt__(b)` | `b.__gt__(a)` |
| `a <= b` | `a.__le__(b)` | `b.__ge__(a)` |
| `a > b` | `a.__gt__(b)` | `b.__lt__(a)` |
| `a >= b` | `a.__ge__(b)` | `b.__le__(a)` |
| `a == b` | `a.__eq__(b)` | `b.__eq__(a)`, then `a is b` |
| `a != b` | `a.__ne__(b)` | `b.__ne__(a)`; by default, the opposite of `__eq__` |

The right-hand column is **reflection**: `a > b` means the same as `b < a`, so if `a` has no `__gt__` (or doesn't know how to compare), Python asks `b`'s `__lt__`. That's why the warm-up's `>` worked. If neither side knows, Python raises `TypeError`, which is what you want when comparing a runner with a number.

What Python does **not** do is combine methods. Having `__lt__` and `__eq__` doesn't give you `<=`:

As with `__eq__` in Week 29, return `NotImplemented` (not `False`) for types you don't understand. Then Python can try the other side, and you get a clear `TypeError` instead of a quietly wrong `False`.

## Comparing tuples of attributes

Tuples compare item by item: the first items decide, unless they're equal, in which case the second items decide, and so on. That's exactly what most orderings need, so the usual way to write a comparison is to build a **key tuple** and compare those. Use the same tuple for `__eq__` and `__hash__`, and all three agree automatically:

The last line shows why a class helps: compared as **strings**, `"1.10"` is less than `"1.9"` (because `"1" < "9"`), but as versions it's greater.

To reverse part of an order (a higher score should come first, but ties go alphabetically), negate the number in the key: `(-self.points, self.name)`. That works for numbers, not strings; for a string you'd use a key function with `reverse=True`, or two sorts (sorting is stable, so a second sort keeps the order of the first among equal items).

## total_ordering

Writing four ordering methods that all say the same thing is tedious and error-prone. `functools.total_ordering` writes the missing ones for you: define `__eq__` and **one** of `__lt__`, `__le__`, `__gt__` or `__ge__`, and put `@total_ordering` above the class. (It's a *class decorator*, like `@property` but for a whole class; Phase 4 explains how decorators work.)

Notice that the grades' order isn't alphabetical (`"A*" < "A"` as strings), which is exactly why the class carries its own `ORDER`. `total_ordering` costs a little speed, which only matters in very large sorts; you can always write the methods by hand later.

## Natural order or key function?

Give a class comparison methods only when it has **one obvious order** that everyone would agree on: versions, grades, amounts of money, dates and times, playing-card ranks. Then `sorted(things)` reads naturally.

Many objects can be sorted in several sensible ways. Football teams by points, by goals, by name; books by title, author, year. For those, don't pick one order and build it in. Sort with a **key function** at the place where you need the order:

`operator.attrgetter("year")` is a ready-made key function that does `lambda b: b.year`; `attrgetter("author", "year")` gives a tuple of both. Key functions also work with objects that can't be compared at all, and they never surprise anyone who reads `a < b` in your code.

## Orders that make sense

Python trusts your comparisons. `sorted`, `min`, `max`, `bisect` and sets don't check them, so they must behave like an order:

- for any two objects, exactly **one** of `a < b`, `a == b`, `a > b` is true;
- if `a < b` and `b < c`, then `a < c`.

Comparing the same key tuple in every method guarantees both. Here's what happens when `__lt__` and `__eq__` look at different things:

`a` and `b` aren't equal, but neither is less than the other. To `sorted`, they're tied, so they come out in whatever order they went in, and any code that assumes "neither is less, so they're equal" (as searching a sorted list with `bisect` does) gets the wrong answer. Bugs like this are hard to find, because nothing crashes. Decide what "the same" means once, put it in `_key`, and use it everywhere.

## Practice

## Quiz

## Daily challenge

## Before you go

For each class you've written this week and last, decide: does it have one obvious order, several, or none? Give an order only to the first kind. Tomorrow: callable objects, which can be called with brackets like functions while remembering things between calls.